In [ ]:
from pathlib import Path
from fastkaggle import iskaggle, setup_comp

# Set these only when you want to send your results to your Kaggle account.
submit_to_kaggle = False
publish_to_kaggle = False

## Getting set up

In [ ]:
comp = 'paddy-disease-classification'

path = Path(comp)
if iskaggle or not (path/'train_images').is_dir():
    path = setup_comp(comp)

In [ ]:
path

In [ ]:
from fastai.vision.all import *
set_seed(42)

path.ls()

## Looking at the data

In [ ]:
trn_path = path/'train_images'
files = get_image_files(trn_path)

In [ ]:
img = PILImage.create(files[0])
print(img.size)
img.to_thumb(128)

In [ ]:
from fastcore.parallel import *

def f(o): return PILImage.create(o).size
sizes = parallel(f, files, n_workers=8)
pd.Series(sizes).value_counts()

In [ ]:
dls = ImageDataLoaders.from_folder(trn_path, valid_pct=0.2, seed=42,
    item_tfms=Resize(480, method='squish'),
    batch_tfms=aug_transforms(size=128, min_scale=0.75))

dls.show_batch(max_n=6)

## Our first model

In [ ]:
learn = vision_learner(dls, 'resnet26d', metrics=error_rate, path='.')
# CUDA uses the lecture precision. The CPU kernel uses float32.
if torch.cuda.is_available(): learn = learn.to_fp16()

In [ ]:
learn.lr_find(suggest_funcs=(valley, slide))

In [ ]:
learn.fine_tune(3, 0.01)

## Submitting to Kaggle

In [ ]:
ss = pd.read_csv(path/'sample_submission.csv')
ss

In [ ]:
tst_files = get_image_files(path/'test_images').sorted()
tst_dl = dls.test_dl(tst_files)

In [ ]:
probs,_,idxs = learn.get_preds(dl=tst_dl, with_decoded=True)
idxs

In [ ]:
dls.vocab

In [ ]:
mapping = dict(enumerate(dls.vocab))
results = pd.Series(idxs.numpy(), name="idxs").map(mapping)
results

In [ ]:
ss['label'] = results
ss.to_csv('subm.csv', index=False)
print('\n'.join(Path('subm.csv').read_text().splitlines()[:10]))

In [ ]:
if submit_to_kaggle and not iskaggle:
    from kaggle import api
    api.competition_submit('subm.csv', 'initial rn26d 128px', comp)

## Conclusion

## Addendum

In [ ]:
# Optional: run from the directory containing this notebook.
if publish_to_kaggle and not iskaggle:
    import json, shutil
    from tempfile import TemporaryDirectory
    from fastkaggle import nb_meta
    from kaggle import api

    filename = '08-first-steps-road-to-the-top-part-1.ipynb'
    meta = nb_meta(api.config_values['username'], 'first-steps-road-to-the-top-part-1', 'First Steps: Road to the Top, Part 1',
                   filename, competition=comp, private=False, gpu=True)
    with TemporaryDirectory() as folder:
        shutil.copyfile(filename, Path(folder)/filename)
        (Path(folder)/'kernel-metadata.json').write_text(json.dumps(meta), encoding='utf-8')
        api.kernels_push(folder)